# 18 — Tuning CatBoost + te_origin multi-lissage (battre 08)

Seul levier solo non testé : régler les hyperparamètres (régularisation pour signal bruité) et
donner `te_origin` à plusieurs lissages. Réf 08 : **recent2 0.3662 | last 0.3607**.
Critère de décision = **recent2** (proxy privé fiable). Boussole : LB ≈ last − 0.004.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6; WINDOWS = (5, 10, 20); SMOOTHINGS = [5, 30, 100]
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X
def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)
def feats_train(df, ref, smooths):
    X = base_build(df, ref)
    for s in smooths:
        X[f"te_origin_{s}"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=s)
    return X
def feats_apply(df, ref, smooths):
    X = base_build(df, ref)
    for s in smooths:
        mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=s)
        X[f"te_origin_{s}"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X
def make_cat(params):
    from catboost import CatBoostClassifier
    base = dict(loss_function="Logloss", eval_metric="PRAUC", random_seed=42, verbose=False)
    return CatBoostClassifier(**{**base, **params})

## Balayage : chaque config jugée sur recent2 vs 0.3662

In [ ]:
CONFIGS = {
    "08_ref (d6,lr05,600,sm30)": (dict(depth=6, learning_rate=0.05, iterations=600), [30]),
    "multi-smooth (5,30,100)":  (dict(depth=6, learning_rate=0.05, iterations=600), SMOOTHINGS),
    "reg (l2=10, rs=2)":         (dict(depth=6, learning_rate=0.05, iterations=600, l2_leaf_reg=10, random_strength=2), SMOOTHINGS),
    "deep (d8, l2=6, 800)":      (dict(depth=8, learning_rate=0.04, iterations=800, l2_leaf_reg=6), SMOOTHINGS),
    "slow (d6, lr03, 1500, l2=6)": (dict(depth=6, learning_rate=0.03, iterations=1500, l2_leaf_reg=6), SMOOTHINGS),
}
def run_cv(params, smooths):
    oof = np.zeros(len(train)); pf = []
    for tr_idx, va_idx in folds_full:
        tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]; ref = train.iloc[tr_op]
        m = make_cat(params).fit(feats_train(train.iloc[tr_op], ref, smooths), y_all[tr_op])
        oof[va_op] = m.predict_proba(feats_apply(train.iloc[va_op], ref, smooths))[:, 1]
        pf.append(evaluate_ap(y_all[va_op], oof[va_op]))
    return pf, oof

best = (None, -1, None, None)
for name, (params, smooths) in CONFIGS.items():
    pf, oof = run_cv(params, smooths)
    r2 = np.mean(pf[-2:])
    flag = "  <-- bat 08" if r2 > 0.3662 else ""
    print(f"{name:32s} recent2 {r2:.4f} | last {pf[-1]:.4f}{flag}")
    if r2 > best[1]: best = (name, r2, params, smooths)
print(f"\nMeilleure config : {best[0]}  (recent2 {best[1]:.4f}, réf 08 = 0.3662)")

## Soumission de la meilleure config (si recent2 > 0.3662)

In [ ]:
name, r2, params, smooths = best
ref_full = train.iloc[np.where(op03)[0]]; yf = y_all[op03]
final = make_cat(params).fit(feats_train(ref_full, ref_full, smooths), yf)
te_op = op03_mask(test).to_numpy(); test_op = test.iloc[np.where(te_op)[0]]
proba = final.predict_proba(feats_apply(test_op, ref_full, smooths))[:, 1]
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "18_tuned_cat")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print(f"config={name} | soumission {path} | proba>0 : {int((sub['target']>0).sum())}")
print("Soumets seulement si recent2 > 0.3662 (sinon garde 08).")